In [ ]:
"""
Callset benchmarking and comparison.
"""

import copy
import importlib
import re
import pickle
import warnings
from pathlib import Path
from collections import defaultdict
import time

import numpy as np
import pandas as pd
import matplotlib as mpl
import pyranges as pr
import pysam
import seaborn as sns
import shap
from scipy.spatial import cKDTree
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import os
from typing import Union, Tuple, Optional, List, Any, Literal, Dict


workdir = "/vast/ai_projects/Papenfuss_SVEnsemble2"
os.chdir(workdir)
os.getcwd()

import svensemble.vcfparse as vcfp # noqa: E402 (flake8)
import svensemble.breakends as bnds
import svensemble.breakpoints as bps
import svensemble.events as evts
import svensemble.caller_specific as caller_specific
import svensemble.benchmark as benchmark
import svensemble._caller_params as caller_params
import svensemble._sample_params as sample_params
import svensemble.chunk_match as chunk_match
import svensemble.matched_breakpoints as matchedbps
import svensemble.utils as sv_utils
import svensemble.importances as importances

importlib.reload(vcfp)
importlib.reload(bnds)
importlib.reload(bps)
importlib.reload(evts)
importlib.reload(caller_specific)
importlib.reload(benchmark)
importlib.reload(caller_params)
importlib.reload(sample_params)
importlib.reload(chunk_match)
importlib.reload(matchedbps)
importlib.reload(sv_utils)

pd.set_option('display.max_columns', None)

model_name = 'unbiased'


In [ ]:
# When converting to a py script.
# Input:
# - sample name
# - model name

# - validation set path - ADD LATER

# - sv types - how??? For now use presets.
#   - DEL
#   - ALL
# - sv lengths. Will need 2 separate values for upper and lower bounds.

In [ ]:
training_sets =  {
    'gridss': ['gridss'],
    'manta_candidate': ['manta_candidate'],
    'manta_diploid': ['manta_diploid'],
    'gridss+manta_candidate': ['gridss', 'manta_candidate'],
    'gridss+manta_diploid': ['gridss', 'manta_diploid']
}
training_sets = caller_params.training_set_presets['default']

sample_names = ['NA12878_30x', 'NA12878', 'CHM13', 'HG002']
# sample_names = ['NA12878_30x', 'CHM13', 'NA12878']
# sample_names = ['NA12878_30x']

model_name = 'unbiased'

val_vcf_paths = {
    'kosugi': '/home/users/allstaff/shikov.v/projects/Papenfuss_SVEnsemble2/data/Kosugi_validation/NA12878_DGV-2016_LR-assembly.hg38.bed',
    'chm13_val': '/home/users/allstaff/shikov.v/projects/Papenfuss_SVEnsemble2/data/CHM13_validation/supporting_variants_for_nstd137.csv',
    'hg002_val': '/home/users/allstaff/shikov.v/projects/Papenfuss_SVEnsemble2/data/HG002_validation/HG002_validation.vcf.gz'
}

svtypes = ['DEL']
svlens = (50, 1000)

verbose = True


In [ ]:
samples = {}
for sample_name in sample_names:
    samples.update(
        sv_utils.create_config_dict(sample_name)
    )
    samples[sample_name]['val_set'] = sample_params.sample_val_sets[sample_name]

In [ ]:
samples['NA12878']['val_set']

In [ ]:
# Val set preprocessing
if 'kosugi' in val_vcf_paths.keys():
    benchmark.preprocess_kosugi_val_set(sample=samples['NA12878'], vcf_path=val_vcf_paths['kosugi'])
if 'chm13_val' in val_vcf_paths.keys():
    benchmark.preprocess_chm13_val_set(sample=samples['CHM13'], vcf_path=val_vcf_paths['chm13_val'])
if 'hg002_val' in val_vcf_paths.keys():
    benchmark.preprocess_hg002_val_set(sample=samples['HG002'], vcf_path=val_vcf_paths['hg002_val'])

In [ ]:
samples = {}
for sample_name in sample_names:
    benchmark.preprocess_sample_for_benchmark(sample_name, model_name, save_pickle=True)

In [ ]:
# Scripts for benchmarking itself: overlapping sample svensemble calls as well as original callsets with the corresponding validation set.

# Here we create a validation dictinary containing
# Validation dictionary is a nested dictionary, with upper level keys being the sample names
# For each sample, a dictionary exists with for each callset as well as for each svensemble2 prediction




In [ ]:
for sample_name in sample_names:
    benchmark.benchmark_sample(
        sample_name,
        model_name,
        svtypes = svtypes,
        svlens = svlens,
        val_match_margin=500,
        save_pickle=True,
        verbose=True
    )

In [ ]:

def plot_cipos_boxplot(df, ax=None, type='box', title=None):
    plot_type_dispatch = {
        'box': sns.boxplot,
        'violin': sns.violinplot
    }
    try:
        plot_func = plot_type_dispatch[type]
    except KeyError:
        raise ValueError(f"Invalid plot type: {type}. Must be one of: {', '.join(plot_type_dispatch.keys())}")
    # Extract the relevant DataFrame
    cipos_cols = [col for col in df.columns if 'INFO_CIPOS' in col]
    df = df[cipos_cols + ['is_in_val']]

    # Take absolute values and prevent log(0). Log-transform the values
    df = df.abs()
    for df_col in df.columns[:-1]:
        df.loc[df[df_col] == 0, df_col] = 1
        df.loc[:, df_col] = np.log(df[df_col])

    df_melted = df.melt(id_vars=['is_in_val'], value_vars=df.columns[:-1])

    # Create a boxplot grouped by 'is_in_val'
    fig, ax = plt.subplots(figsize=(10, 6))

    plot_func(x='variable', y='value', hue='is_in_val', data=df_melted, ax=ax)
    if title:
        ax.set_title(title)


In [ ]:
val_dict_path = f'/home/users/allstaff/shikov.v/projects/Papenfuss_SVEnsemble2/data/{sample_name}/sv_calling/svensemble2/{sample_name}.{model_name}.{'-'.join(svtypes)}.{svlens[0]}:{svlens[1]}.val_dict.pkl'
val_dict = pickle.load(open(val_dict_path, 'rb'))


In [ ]:
sample_name = 'NA12878_30x'
benchmark.plot_reverse_roc_curves(val_dict, filter_pass=False, title=sample_name)

In [ ]:
sample_name = 'NA12878_30x'
benchmark.plot_roc_curves(val_dict, filter_pass=False, title=sample_name)

In [ ]:
for sample_name in val_dict.keys():
    benchmark.plot_roc_curves(val_dict[sample_name], filter_pass=False, title=sample_name, svtype='DEL')

In [ ]:
# Shap stuff
sample_name = 'NA12878_30x'
training_set_name = 'gridss+manta_diploid'
model_name = 'unbiased'

samples_pkl_path = 'model/saved/samples.pkl'
val_dict_pkl_path = 'model/saved/val_dict.pkl'




X, y = importances.load_svensemble2_val_Xy_from_pickles(samples_pkl_path, val_dict_pkl_path, sample_name, training_set_name, verbose=True)

samples = pickle.load(open(samples_pkl_path, 'rb'))
model = samples[sample_name]['pred_dicts'][training_set_name]['RF_fit']

In [ ]:
exp = shap.TreeExplainer(model)
sv = exp.shap_values(X)

In [ ]:
sv = pickle.load(open('/home/users/allstaff/shikov.v/projects/Papenfuss_SVEnsemble2/model/temp/shap_values.pkl', 'rb'))

In [ ]:
np.shape(sv)

In [ ]:
dir(sv[:,:,1])

In [ ]:
sv[:,:,1].abs

In [ ]:
shap.plots.bar(sv[:,:,1].abs.max(0))


In [ ]:
sv.output_names

In [ ]:
shap.plots.waterfall(sv[:,:,1][10])

In [ ]:
shap.plots.beeswarm(sv[:,:,0], max_display=20)


In [ ]:
shap.plots.beeswarm(sv[:,:,1], order=shap.Explanation.abs.max(0), max_display=30)

In [ ]:
shap.plots.scatter(sv[:,:,1][:, "manta_diploid_SAMPLE_SR_REF"])


In [ ]:
shap.plots.scatter(sv[:,:,1][:, "manta_diploid_SAMPLE_PR_REF"])


In [ ]:
ax = shap.plots.scatter(sv[:,:,1][:, "gridss_INFO_RASQ_bnd2"])


In [ ]:
shap.plots.initjs()
shap.plots.force(sv[:,:,1][:1000])


In [ ]:
samples['NA12878'].keys()

In [ ]:
samples['NA12878']['matched_bps']['gridss+manta_diploid'].iloc[685]

In [ ]:
benchmark.plot_roc_curves({val_set_name: val_set for val_set_name, val_set in val_dict['NA12878_30x'].items() if val_set_name in ['manta_diploid', 'svensemble2_manta_diploid']}, filter_pass=False, title='NA12878_30x')

In [ ]:
benchmark.plot_roc_curves(val_dict['NA12878'], filter_pass=True, title='NA12878')

In [ ]:
benchmark.plot_roc_curves(val_dict['CHM13'], filter_pass=True, title='CHM13')

In [ ]:
benchmark.plot_roc_curves(val_dict['HG002'], ax=None, use_counts=True, title='HG002')

In [ ]:
benchmark.plot_precision_recall_curves(val_dict['HG002'])

In [ ]:
samples['NA12878']['pred_dicts']

In [ ]:
import sklearn.model_selection as skm

verbose = True
sample_config = samples['NA12878']
reference_sample_config = sv_utils.create_config_dict('gnomad', is_reference=True)['gnomad']
training_set_name = 'gridss+manta_diploid'



In [ ]:
sample_name = 'NA12878'
training_set_name = 'gridss+manta_diploid'

svensemble2_model = samples[sample_name]['pred_dicts'][training_set_name]['RF_fit']
svensemble2_splits = load_svensemble2_train_test_splits(samples[sample_name], sv_utils.create_config_dict('gnomad', is_reference=True)['gnomad'], training_set_name, verbose=True)
svensemble2_X_val = svensemble2_splits['X_val']
svensemble2_y_val = svensemble2_splits['y_val']



In [ ]:
svensemble2_importances = svensemble2_premutation_importance(svensemble2_model, svensemble2_X_val, svensemble2_y_val, n_repeats=10, random_state=42)

In [ ]:
svensemble2_importances

In [ ]:
fig, ax = plt.subplots()
svensemble2_importances.sort_values(ascending=False).plot(ax=ax, kind='barh')
ax.set_title("Mean accuracy decrease")
ax.set_ylabel("Feature importances using permutation on full model")

fig.tight_layout()
fig.set_size_inches((5,10))
plt.show()
# plt.show()
# plt.savefig('figs/featureimportance_perm.enoto.png')